# 01 Kalibrierung

**Eingabe**: Roh-FITS (Light, Dark, Flat, Bias) — aus einem Siril-Workspace oder beliebigen Ordnern  
**Ausgabe**: Pfad zu kalibrierten FITS-Einzelbildern für Notebook 02

### Ablauf
1. Siril-Workspace wählen → Unterordner werden automatisch erkannt
2. Kalibrierung: **Siril** (GUI oder Skript, empfohlen) oder **ccdproc** (Python)
3. Pfad der kalibrierten Bilder für Notebook 02 festlegen

Siril-Standardstruktur: `lights/`, `darks/`, `flats/`, `biases/` (Groß-/Kleinschreibung egal).

## Vorbedingungen prüfen

Diese Zelle prüft alle Abhängigkeiten bevor das Notebook gestartet wird:
Python-Pakete, optionale externe Programme und den Kernel-Pfad.

In [ ]:
import sys, shutil, importlib
from IPython.display import display, HTML

_checks = []

def _chk(label, ok, detail=""):
    _checks.append((label, ok, detail))

# --- Python-Pakete ---
_PACKAGES = [
    ("ipyfilechooser", "ipyfilechooser"),
    ("ipywidgets",     "ipywidgets"),
    ("astropy",        "astropy"),
    ("numpy",          "numpy"),
    ("matplotlib",     "matplotlib"),
    ("astroquery",     "astroquery"),
]
for name, mod in _PACKAGES:
    try:
        m = importlib.import_module(mod)
        ver = getattr(m, "__version__", "?")
        _chk(f"pkg: {name}", True, ver)
    except ImportError:
        _chk(f"pkg: {name}", False, "nicht installiert — pip install " + name)

# --- Externe Programme (optional) ---
_BINS = [
    ("ASTAP",  ["/Applications/ASTAP.app/Contents/MacOS/astap"]),
    ("Siril",  ["/Applications/Siril.app/Contents/MacOS/siril",
                "/usr/bin/siril", shutil.which("siril") or ""]),
]
for prog, paths in _BINS:
    found = next((p for p in paths if p and shutil.os.path.exists(p)), None)
    _chk(f"bin: {prog}", found is not None,
         found if found else "nicht gefunden (optional — nur für echte Daten nötig)")

# --- Kernel-Info ---
_chk("Kernel", True, f"Python {sys.version.split()[0]}  —  {sys.executable}")

# --- HTML-Ausgabe ---
rows = ""
all_ok = True
for label, ok, detail in _checks:
    icon  = "✓" if ok else "✗"
    color = "#4caf50" if ok else "#f44336"
    if not ok:
        all_ok = False
    rows += (
        f'<tr><td style="padding:3px 10px;color:{color};font-weight:bold">{icon}</td>'
        f'<td style="padding:3px 10px">{label}</td>'
        f'<td style="padding:3px 10px;color:#888;font-size:12px">{detail}</td></tr>'
    )

status_color = "#4caf50" if all_ok else "#f44336"
status_text  = "Alle Vorbedingungen erfüllt — Notebook kann gestartet werden." \
               if all_ok else "⚠ Mindestens eine Vorbedingung fehlt (s. ✗-Zeilen)."

display(HTML(f"""
<table style="border-collapse:collapse;font-family:monospace;font-size:13px;margin:6px 0">
  <tr style="background:#1e1e2e;color:#cdd6f4">
    <th colspan="3" style="padding:6px 14px;text-align:left">
      Preflight-Check — Notebook 01 Kalibrierung
    </th>
  </tr>
  {rows}
  <tr><td colspan="3" style="padding:6px 10px;color:{status_color};font-weight:bold">
    {status_text}
  </td></tr>
</table>
"""))

## Imports

Lädt die benötigten Bibliotheken: `ipyfilechooser` für den Ordner-Picker,
`ipywidgets` für die Text-Eingabefelder, `astropy.io.fits` für den FITS-Header-Check.

In [3]:
from pathlib import Path
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import display
from astropy.io import fits
import json

## 1 — Siril-Workspace wählen

Navigiere zum Ordner, den du in Siril als Arbeitsverzeichnis eingestellt hast
(Siril → Datei → Einstellungen → Verzeichnisse). Die Unterordner `lights/`, `darks/`,
`flats/` und `biases/` werden automatisch erkannt.

In [5]:
# Siril-Workspace-Ordner wählen
ws_chooser = FileChooser(
    path=str(Path.home()),
    title="Siril-Workspace (Ordner mit lights/, darks/, flats/, biases/)",
    show_only_dirs=True,
    show_hidden=False,
)
display(ws_chooser)

FileChooser(path='/Users/apos', filename='', title='Siril-Workspace (Ordner mit lights/, darks/, flats/, biase…

## 2 — Ordnerstruktur erkennen und anzeigen

Zelle ausführen **nachdem** oben ein Ordner gewählt wurde.

In [6]:
# Siril-Vorgabenamen je Kategorie (Groß-/Kleinschreibung)
_SIRIL_DIRS = {
    "lights":  ["lights", "Light", "Lights", "light", "LIGHTS"],
    "darks":   ["darks",  "Dark",  "Darks",  "dark",  "DARKS"],
    "flats":   ["flats",  "Flat",  "Flats",  "flat",  "FLATS"],
    "biases":  ["biases", "Bias",  "bias",   "Biases","BIAS", "bias_frames"],
    "process": ["process","Process","PROCESS"],
}
_FITS_GLOB = ["*.fits", "*.fit", "*.fts", "*.FITS", "*.FIT", "*.FTS"]

def _fits_in(folder):
    files = []
    for pat in _FITS_GLOB:
        files.extend(folder.glob(pat))
    return sorted(set(files))

def detect_siril_workspace(workspace):
    ws = Path(workspace)
    result = {}
    for key, names in _SIRIL_DIRS.items():
        for name in names:
            p = ws / name
            if p.is_dir():
                files = _fits_in(p)
                result[key] = {"path": p, "count": len(files), "files": files}
                break
        else:
            result[key] = {"path": None, "count": 0, "files": []}
    return result

if not ws_chooser.selected:
    raise RuntimeError("Bitte zuerst oben einen Ordner wählen und dann diese Zelle erneut ausführen.")

WORKSPACE = Path(ws_chooser.selected)
dirs = detect_siril_workspace(WORKSPACE)

print(f"Workspace: {WORKSPACE}\n")
print(f"{'Kategorie':<10} {'Ordner':<40} {'FITS'}")  
print("-" * 60)
for key, info in dirs.items():
    folder_str = str(info["path"].relative_to(WORKSPACE)) if info["path"] else "— nicht gefunden —"
    print(f"{key:<10} {folder_str:<40} {info['count']}")

Workspace: /Users/apos/Nextcloud/WORKSPACE/ASTRO/PHOTOMETRIE_PY/demo/siril_workspace

Kategorie  Ordner                                   FITS
------------------------------------------------------------
lights     lights                                   20
darks      darks                                    5
flats      flats                                    5
biases     biases                                   5
process    process                                  20


## 3 — Pfade manuell korrigieren (optional)

Falls die automatische Erkennung einen Ordner nicht gefunden hat oder
du von der Siril-Standardstruktur abweichst, hier den Pfad eintragen.

In [9]:
def _path_or(detected, manual):
    """Gibt den manuellen Pfad zurück wenn angegeben, sonst den automatisch erkannten."""
    if manual.strip():
        p = Path(manual.strip())
        return p if p.is_dir() else None
    return detected

style = {"description_width": "80px"}
layout = widgets.Layout(width="500px")

w_lights  = widgets.Text(description="lights:",  placeholder="leer lassen = auto", style=style, layout=layout)
w_darks   = widgets.Text(description="darks:",   placeholder="leer lassen = auto", style=style, layout=layout)
w_flats   = widgets.Text(description="flats:",   placeholder="leer lassen = auto", style=style, layout=layout)
w_biases  = widgets.Text(description="biases:",  placeholder="leer lassen = auto", style=style, layout=layout)
w_cal_out = widgets.Text(description="cal-out:", placeholder="Ausgabe kalibrierter FITS (leer = process/)",
                         style=style, layout=layout)

# Vorbefüllen mit erkannten Pfaden
for w, key in [(w_lights, "lights"), (w_darks, "darks"),
               (w_flats, "flats"), (w_biases, "biases")]:
    if dirs[key]["path"]:
        w.value = str(dirs[key]["path"])
if dirs["process"]["path"]:
    w_cal_out.value = str(dirs["process"]["path"])

display(widgets.VBox([
    widgets.HTML("<b>Pfade (bei Bedarf anpassen):</b>"),
    w_lights, w_darks, w_flats, w_biases,
    widgets.HTML("<b>Ausgabe kalibrierte FITS:</b>"),
    w_cal_out,
]))

## 4 — Pfade übernehmen und FITS-Header prüfen

Zelle ausführen **nachdem** oben alle Pfade gesetzt sind.

In [10]:
DIR_LIGHTS = _path_or(dirs["lights"]["path"],  w_lights.value)
DIR_DARKS  = _path_or(dirs["darks"]["path"],   w_darks.value)
DIR_FLATS  = _path_or(dirs["flats"]["path"],   w_flats.value)
DIR_BIASES = _path_or(dirs["biases"]["path"],  w_biases.value)
DIR_CAL    = Path(w_cal_out.value.strip()) if w_cal_out.value.strip() \
             else WORKSPACE / "process"
DIR_CAL.mkdir(parents=True, exist_ok=True)

print("Aktive Pfade:")
for label, p in [("lights", DIR_LIGHTS), ("darks", DIR_DARKS),
                 ("flats", DIR_FLATS), ("biases", DIR_BIASES), ("cal-out", DIR_CAL)]:
    status = "✓" if (p and p.exists()) else "✗ FEHLT"
    print(f"  {label:<10} {status}  {p}")

# FITS-Header des ersten Light-Frames prüfen
lights = _fits_in(DIR_LIGHTS) if DIR_LIGHTS else []
print(f"\n{len(lights)} Light-Frames gefunden.")
if lights:
    with fits.open(lights[0]) as hdul:
        h = hdul[0].header
    chk_keys = ["DATE-OBS", "EXPTIME", "FILTER", "OBJECT",
                "XBINNING", "GAIN", "CCD-TEMP", "SITELAT", "SITELONG"]
    print("\nHeader-Check (erster Light-Frame):")
    for k in chk_keys:
        val = h.get(k, "— fehlt —")
        flag = "  " if k in h else " !"
        print(f"  {flag} {k:<12} {val}")

Aktive Pfade:
  lights     ✓  /Users/apos/Nextcloud/WORKSPACE/ASTRO/PHOTOMETRIE_PY/demo/siril_workspace/lights
  darks      ✓  /Users/apos/Nextcloud/WORKSPACE/ASTRO/PHOTOMETRIE_PY/demo/siril_workspace/darks
  flats      ✓  /Users/apos/Nextcloud/WORKSPACE/ASTRO/PHOTOMETRIE_PY/demo/siril_workspace/flats
  biases     ✓  /Users/apos/Nextcloud/WORKSPACE/ASTRO/PHOTOMETRIE_PY/demo/siril_workspace/biases
  cal-out    ✓  /Users/apos/Nextcloud/WORKSPACE/ASTRO/PHOTOMETRIE_PY/demo/siril_workspace/process

20 Light-Frames gefunden.

Header-Check (erster Light-Frame):
     DATE-OBS     2024-09-02T19:26:33.353
     EXPTIME      120.0
     FILTER       V
     OBJECT       Delta Cep
     XBINNING     1
     GAIN         0
     CCD-TEMP     -15.0
     SITELAT      48.3
     SITELONG     14.3


## 5 — Kalibrierung

**Option A — Siril (empfohlen)**: Kalibrierung in Siril durchführen, Ergebnis landet in `process/`.
Dann oben unter *cal-out* den `process/`-Pfad eintragen und Schritt 4 erneut ausführen.

**Option B — ccdproc (Python)**: Zelle unten aktivieren und anpassen.

In [ ]:
# Option B: ccdproc — noch nicht implementiert
# from ccdproc import ImageFileCollection, Combiner, ccd_process
# ...
print("Kalibrierung: Siril-Ergebnis verwenden oder ccdproc-Block aktivieren.")

## 6 — Konfiguration für Notebook 02 speichern

In [11]:
cfg_path = WORKSPACE / "session_config.json"
session_cfg = {
    "workspace":   str(WORKSPACE),
    "dir_lights":  str(DIR_LIGHTS)  if DIR_LIGHTS  else None,
    "dir_darks":   str(DIR_DARKS)   if DIR_DARKS   else None,
    "dir_flats":   str(DIR_FLATS)   if DIR_FLATS   else None,
    "dir_biases":  str(DIR_BIASES)  if DIR_BIASES  else None,
    "dir_cal":     str(DIR_CAL),
    "n_lights":    len(lights),
}
cfg_path.write_text(json.dumps(session_cfg, indent=2, ensure_ascii=False))
print(f"Konfiguration gespeichert: {cfg_path}")
print("→ Notebook 02 liest diese Datei automatisch.")

Konfiguration gespeichert: /Users/apos/Nextcloud/WORKSPACE/ASTRO/PHOTOMETRIE_PY/demo/siril_workspace/session_config.json
→ Notebook 02 liest diese Datei automatisch.
